# Tetris mit Reinforcement Learning - GPU Version

## GPU-beschleunigtes Training
- **32 Umgebungen** (statt 16)
- **600.000 Trainingsschritte** (statt 300.000)
- **CUDA GPU Support** für 5-10x schnelleres Training

## 1. GPU Check

In [1]:
import torch
print("="*60)
print("GPU / CUDA CHECK")
print("="*60)
if torch.cuda.is_available():
    print("✓ GPU wird verwendet!")
    print(f"  - GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  - GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  - PyTorch CUDA Version: {torch.version.cuda}")
else:
    print("⚠ Keine GPU gefunden - Training läuft auf CPU")
print("="*60)

GPU / CUDA CHECK
⚠ Keine GPU gefunden - Training läuft auf CPU


## 2. Imports

In [2]:
import numpy as np
import socket
import cv2
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
import subprocess
import os
import shutil
import glob
import imageio
from IPython.display import Image, display
import time

from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.env_checker import check_env

print("✓ Alle Imports erfolgreich")

✓ Alle Imports erfolgreich


## 3. Tetris Server starten

In [3]:
print("Starte Tetris Server...")
server_process = subprocess.Popen(["java", "-jar", "TetrisTCPserver_v0.6.jar"])
time.sleep(3)
print("✓ Tetris Server gestartet")

Starte Tetris Server...
Tetris TCP server is listening at 10612
✓ Tetris Server gestartet


## 4. TetrisEnv Klasse

In [4]:
class TetrisEnv(gym.Env):
    """Gymnasium Umgebung für Tetris."""

    metadata = {"render_modes": ["human"], "render_fps": 20}
    N_DISCRETE_ACTIONS = 5
    IMG_HEIGHT = 200
    IMG_WIDTH = 100
    IMG_CHANNELS = 3

    def __init__(self, host_ip="127.0.0.1", host_port=10612):
        super().__init__()
        self.action_space = spaces.Discrete(self.N_DISCRETE_ACTIONS)
        self.observation_space = spaces.Box(
            low=0, high=255,
            shape=(self.IMG_HEIGHT, self.IMG_WIDTH, self.IMG_CHANNELS),
            dtype=np.uint8
        )
        self.server_ip = host_ip
        self.server_port = host_port
        self.client_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.client_sock.connect((self.server_ip, self.server_port))

    def step(self, action):
        if action == 0:
            self.client_sock.sendall(b"move -1\n")
        elif action == 1:
            self.client_sock.sendall(b"move 1\n")
        elif action == 2:
            self.client_sock.sendall(b"rotate 0\n")
        elif action == 3:
            self.client_sock.sendall(b"rotate 1\n")
        elif action == 4:
            self.client_sock.sendall(b"drop\n")

        terminated, lines, height, holes, observation = self.get_tetris_server_response(self.client_sock)
        self.observation = observation

        reward = 0
        if action == 4:
            reward += 5
        if height > self.height:
            reward -= (height - self.height) * 5
        if holes < self.holes:
            reward += (self.holes - holes) * 10
        if lines > self.lines_removed:
            reward = reward + (lines - self.lines_removed) * 1000
            self.lines_removed = lines

        self.holes = holes
        self.height = height
        self.lifetime += 1
        truncated = False
        info = {'removed_lines': self.lines_removed, 'lifetime': self.lifetime}
        return (observation, reward, terminated, truncated, info)

    def reset(self, seed=None, options=None):
        self.client_sock.sendall(b"start\n")
        terminated, lines, height, holes, observation = self.get_tetris_server_response(self.client_sock)
        self.observation = observation
        self.reward = 0
        self.lines_removed = 0
        self.holes = 0
        self.height = 0
        self.lifetime = 0
        info = {}
        return observation, info

    def render(self):
        pass

    def close(self):
        self.client_sock.close()

    def get_tetris_server_response(self, sock):
        is_game_over = (sock.recv(1) == b'\x01')
        removed_lines = int.from_bytes(sock.recv(4), 'big')
        height = int.from_bytes(sock.recv(4), 'big')
        holes = int.from_bytes(sock.recv(4), 'big')
        img_size = int.from_bytes(sock.recv(4), 'big')
        img_png = sock.recv(img_size)
        nparr = np.frombuffer(img_png, np.uint8)
        np_image = cv2.imdecode(nparr, -1)
        return is_game_over, removed_lines, height, holes, np_image

print("✓ TetrisEnv Klasse definiert")

✓ TetrisEnv Klasse definiert


## 5. Umgebung überprüfen

In [5]:
print("Überprüfe die Tetris Umgebung...")
env = TetrisEnv()
check_env(env)
env.close()
print("✓ Umgebung ist korrekt konfiguriert")

Überprüfe die Tetris Umgebung...
Client has joined the game
✓ Umgebung ist korrekt konfiguriert
Client has exited the game


## 6. GPU-Training mit 32 Umgebungen

**GPU Unterschiede:**
- 32 parallele Umgebungen
- 600.000 Trainingsschritte
- device='cuda' für GPU

In [ ]:
print("Erstelle 32 parallele Tetris Umgebungen...")
vec_env = make_vec_env(TetrisEnv, n_envs=32)
print("✓ Umgebungen erstellt\n")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}\n")

model = A2C(
    "CnnPolicy",
    vec_env,
    verbose=1,
    device=device,
    tensorboard_log="./sb3_log_gpu/"
)

print("Training läuft...\n")
start_time = time.time()
model.learn(total_timesteps=600000)
training_time = time.time() - start_time

print(f"\n✓ Training fertig: {training_time/60:.1f} Minuten")

Erstelle 32 parallele Tetris Umgebungen...
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
Client has joined the game
✓ Umgebungen erstellt

Device: CPU

Using cpu device
Client has joined the game
Wrapping the env in a VecTransposeImage.

## 7. Testen

In [ ]:
print("Starte Testing...\n")
obs = vec_env.reset()
test_steps = 1500
replay_folder = './replay_gpu'

if os.path.exists(replay_folder):
    shutil.rmtree(replay_folder)

n_env = obs.shape[0]
ep_id = np.zeros(n_env, int)
ep_steps = np.zeros(n_env, int)
cum_reward = np.zeros(n_env)
max_reward = -1e10
max_game_id = 0
max_ep_id = 0
max_rm_lines = 0
max_lifetime = 0
first_rm_lines = 0
first_lifetime = 0
first_game_found = False

for step in range(test_steps):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)

    if step % 300 == 0:
        print(f"Step {step}/{test_steps}")

    for eID in range(n_env):
        cum_reward[eID] += reward[eID]
        folder = f'{replay_folder}/{eID}/{ep_id[eID]}'
        os.makedirs(folder, exist_ok=True)
        fname = folder + '/' + '{:06d}'.format(ep_steps[eID]) + '.png'
        cv2.imwrite(fname, obs[eID])
        ep_steps[eID] += 1

        if done[eID]:
            if cum_reward[eID] > max_reward:
                max_reward = cum_reward[eID]
                max_game_id = eID
                max_ep_id = ep_id[eID]
                max_rm_lines = info[eID]['removed_lines']
                max_lifetime = info[eID]['lifetime']
            
            if not first_game_found and step > 10:
                first_rm_lines = info[eID]['removed_lines']
                first_lifetime = info[eID]['lifetime']
                first_game_found = True
            
            ep_id[eID] += 1
            cum_reward[eID] = 0
            ep_steps[eID] = 0

print("\n✓ Testing fertig")

## 8. Ergebnisse

In [ ]:
print("="*60)
print("TETRIS GPU VERSION - ERGEBNISSE")
print("="*60)
print(f"Device: {device.upper()}")
print(f"Trainingszeit: {training_time/60:.1f} Minuten")
print(f"\nBEST SPIEL:")
print(f"  Reihen: {max_rm_lines}")
print(f"  Dauer: {max_lifetime} Schritte")
print(f"\nERSTES SPIEL:")
print(f"  Reihen: {first_rm_lines}")
print(f"  Dauer: {first_lifetime} Schritte")
print(f"\nVERBESSERUNG:")
print(f"  +{max_rm_lines - first_rm_lines} Reihen")
print(f"  +{max_lifetime - first_lifetime} Schritte")
print("="*60)

## 9. GIFs erstellen

In [ ]:
best_path = f'{replay_folder}/{max_game_id}/{max_ep_id}'
first_path = f'{replay_folder}/0/0'

best_files = sorted(glob.glob(f'{best_path}/*.png'))
if best_files:
    best_images = [imageio.imread(f) for f in best_files]
    imageio.mimsave('best_game_gpu.gif', best_images, loop=0, duration=0.05)
    print(f"✓ best_game_gpu.gif ({len(best_files)} Frames)")

first_files = sorted(glob.glob(f'{first_path}/*.png'))
if first_files:
    first_images = [imageio.imread(f) for f in first_files]
    imageio.mimsave('first_game_gpu.gif', first_images, loop=0, duration=0.05)
    print(f"✓ first_game_gpu.gif ({len(first_files)} Frames)")

## 10. Speichern

In [ ]:
with open('tetris_results_gpu.csv', 'w') as f:
    f.write('game_type,removed_lines,lifetime_steps,training_time_sec\n')
    f.write(f'first,{first_rm_lines},{first_lifetime},{training_time:.1f}\n')
    f.write(f'best,{max_rm_lines},{max_lifetime},{training_time:.1f}\n')

model.save('tetris_a2c_trained_gpu_32env.zip')

print("✓ Ergebnisse gespeichert")
print("  - tetris_results_gpu.csv")
print("  - best_game_gpu.gif")
print("  - first_game_gpu.gif")
print("  - tetris_a2c_trained_gpu_32env.zip")